# Module 1 Bridge -- combined FIRE + LongDR + Tianjin caching

Replaces notebooks 04 and 05's caching cells with a single session: one Drive mount, one
GitHub token, one repo clone, no manual file upload needed on this branch. Builds
all 3 Module 1 output caches (FIRE, LongDR, Tianjin) that notebook 06 needs for the real
combined Module 2 training run.

**Deliberately does NOT run notebook 04's own standalone Module 2 training cell** -- that was
a throwaway 5-epoch FIRE+LongDR-only POC that nothing downstream actually reads (notebook 06
trains its own Module 2 checkpoint from scratch using all 3 sources, and notebook 08 reads
06's checkpoint, not 04's). Skipping it saves a full training run for zero loss.

**After this notebook finishes:** go straight to notebook 06, then 07 and 08.

**`MODULE1_SOURCE`** (first code cell) picks the Module 1 models: `'drgnet_combined'` (default) = notebook 11's FGADR classifier + FGADR + DDR combined lesion segmenters (caches go to `module1_cache_drgnet_comb/`); `'drgnet'` = notebook 11's FGADR classifier + FGADR-only segmenters (`module1_cache_drgnet/`); `'poc'` = notebooks 02/03's (`module1_cache/`). Each setting has its own folders, so nothing is overwritten when you switch.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Free Drive space: the training notebooks save checkpoints and caches here (several GB in total).
# Deleted or overwritten files stay in the Drive Trash and still count -- empty it if space is low.
import shutil as _shutil
try:
    _du = _shutil.disk_usage('/content/drive/MyDrive')
    print(f'Google Drive: {_du.free / 1e9:.1f} GB free of {_du.total / 1e9:.1f} GB')
    if _du.free < 10e9:
        print('⚠ Under 10 GB free on Google Drive -- model files may fail to save. Empty the Trash '
              '(drive.google.com/drive/trash) or free space before training.')
except OSError:
    pass

import getpass, os

GITHUB_TOKEN = getpass.getpass('GitHub Personal Access Token (repo scope): ')

REPO_OWNER = 'Mieka068'
REPO_NAME = 'DRProgression'
REPO_BRANCH = 'AllModuleEdits-PreFinals'  # <-- pre-finals fixes (CODE_FIXES_AllModuleEdits-PreFinals.md); 'mod-2-NonLongIntegration' = the earlier runs
REPO_CODE_SUBDIR = 'M2-DRProgression-VerM-module1-fgadr-poc'

DRIVE_DATA_DIR = '/content/drive/MyDrive/Thesis_Datasets'  # <-- change if you used a different folder
assert os.path.isdir(DRIVE_DATA_DIR), f"Not found: {DRIVE_DATA_DIR}"

# Which Module 1 models the pipeline uses -- set the SAME value in notebooks 04_05 and 06-10:
#   'drgnet' -- notebook 11's DRG-Net replication checkpoints (module1_runs/drgnet_replication/)
#   'poc'    -- notebooks 02/03's original POC checkpoints (module1_runs/)
# Everything built from Module 1's outputs (Module 1 caches, Module 2 runs, figures) gets a
# '_drgnet' suffix for 'drgnet', so the two pipelines never overwrite each other's results.
MODULE1_SOURCE = 'drgnet_combined'
assert MODULE1_SOURCE in ('drgnet_combined', 'drgnet', 'poc'), MODULE1_SOURCE
# New Drive folders for this branch's runs: every Module 1 cache / Module 2 / Module 3 folder below
# gets RUN_SUFFIX, so the mod-2-NonLongIntegration results stay untouched for before/after
# comparison. '' = read/write the earlier folders again.
RUN_SUFFIX = '_prefinals'
RUN_TAG = {'drgnet_combined': '_drgnet_comb', 'drgnet': '_drgnet', 'poc': ''}[MODULE1_SOURCE] + RUN_SUFFIX
MODULE1_RUNS = os.path.join(DRIVE_DATA_DIR, 'module1_runs', *(['drgnet_replication'] if MODULE1_SOURCE != 'poc' else []))
# Notebook 11 on this branch writes its models to drgnet_replication<RUN_SUFFIX>/ (tiled evaluation,
# 200 segmentation epochs, optional EyePACS grader); use those once they exist.
if MODULE1_SOURCE != 'poc' and os.path.isdir(os.path.join(MODULE1_RUNS + RUN_SUFFIX, 'segmentation')):
    MODULE1_RUNS += RUN_SUFFIX
elif MODULE1_SOURCE != 'poc' and RUN_SUFFIX:
    print(f'\u26a0 {MODULE1_RUNS + RUN_SUFFIX} not found -- using the earlier Module 1 models in {MODULE1_RUNS}. '
          'Run notebook 11 on this branch first for the re-trained ones.')
# Lesion segmenters: <MODULE1_RUNS>/segmentation/<SEG_MODEL_PREFIX><lesion>/model_2.pth.tar --
# FGADR + DDR combined (notebook 11) or FGADR only. The classifier is FGADR's either way.
SEG_MODEL_PREFIX = 'comb_models_' if MODULE1_SOURCE == 'drgnet_combined' else 'models_FGADR_NO_TATL_'
MODULE1_CACHE_DIR = os.path.join(DRIVE_DATA_DIR, 'module1_cache' + RUN_TAG)
os.makedirs(MODULE1_CACHE_DIR, exist_ok=True)
print('Module 1 source:', MODULE1_SOURCE, '| models:', MODULE1_RUNS, '| caches:', MODULE1_CACHE_DIR)
TIANJIN_ZIP = os.path.join(DRIVE_DATA_DIR, 'retinal-dr-longitudinal.zip')
assert os.path.isfile(TIANJIN_ZIP), f"Not found: {TIANJIN_ZIP}"


In [ ]:
# Unzip FIRE + LongDR + Tianjin. -n skips files that already exist, so safe/cheap to re-run.
import glob, shutil

os.makedirs('/content/data', exist_ok=True)
%cd /content/data
!unzip -q -n "$DRIVE_DATA_DIR/FIRE_dataset.zip"
!unzip -q -n "$DRIVE_DATA_DIR/LongDRScreening_20150209.zip" -d LongDRScreening_20150209
!unzip -q -n "$TIANJIN_ZIP" -d _tianjin_extract_raw

_manifest_candidates = glob.glob('/content/data/_tianjin_extract_raw/**/corrected_manifest.csv', recursive=True)
assert _manifest_candidates, 'No corrected_manifest.csv found under the extracted Tianjin zip.'
_tianjin_root = os.path.dirname(_manifest_candidates[0])
TIANJIN_DIR = '/content/data/retinal-dr-longitudinal'
if _tianjin_root != TIANJIN_DIR and not os.path.exists(TIANJIN_DIR):
    os.symlink(_tianjin_root, TIANJIN_DIR)

print('FIRE Images present :', os.path.isdir('/content/data/FIRE_dataset/FIRE/Images'))
print('LongDR norm present :', os.path.isdir('/content/data/LongDRScreening_20150209/FundusImagesNormalized'))
print('Tianjin dir present :', os.path.isdir(TIANJIN_DIR))


In [ ]:
# Clone the repo once and install everything both the FIRE/LongDR and Tianjin steps need.
%cd /content
if not os.path.isdir(f'/content/{REPO_NAME}'):
    !git clone --branch {REPO_BRANCH} https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git {REPO_NAME}

REPO_CODE_DIR = f'/content/{REPO_NAME}/{REPO_CODE_SUBDIR}'
assert os.path.isdir(REPO_CODE_DIR), f"Not found: {REPO_CODE_DIR}"

%cd {REPO_CODE_DIR}
!pip install -q pandas openpyxl segmentation-models-pytorch opencv-python-headless


In [ ]:
# Item 7.3: image-size distribution of the three longitudinal datasets (LongDR's and Tianjin's sizes
# were undocumented) -- for the methods section. Header-only reads, a few seconds. Saved to Drive.
%cd {REPO_CODE_DIR}
!python tools/dataset_image_sizes.py fire=/content/data/FIRE_dataset longdr=/content/data/LongDRScreening_20150209 tianjin="{TIANJIN_DIR}" --out "{MODULE1_CACHE_DIR}/dataset_image_sizes.json"

In [ ]:
# Laterality resolution -- Tianjin-only, required before tianjin_dataset.py/apply_to_progression_data.py
# can run over it. Reuses a Drive-persisted copy if one exists (saves a few minutes).
# This branch re-tuned module1/resolve_eye_laterality.py (item 7.1), so it gets its own copy
# (module1_cache<RUN_SUFFIX>/); the settings come from notebook 11's EyePACS validation when it exists.
# _v2: the first file on this branch used the EyePACS validation's convention at only ~61 % accuracy
# (EyePACS mixes mirrored and non-mirrored images) -> 20.6 % of patients had both eyes labelled alike.
LATERALITY_DRIVE_PATH = os.path.join(DRIVE_DATA_DIR, 'module1_cache' + RUN_SUFFIX, 'laterality_resolved_v2.csv')
LATERALITY_VALIDATION = os.path.join(DRIVE_DATA_DIR, 'module1_cache' + RUN_SUFFIX, 'laterality_eyepacs_validation.json')
LATERALITY_LOCAL_PATH = os.path.join(TIANJIN_DIR, 'laterality_resolved.csv')

if os.path.isfile(LATERALITY_DRIVE_PATH):
    shutil.copy(LATERALITY_DRIVE_PATH, LATERALITY_LOCAL_PATH)
    print(f'✓ Reused laterality_resolved.csv from Drive')
else:
    %cd {REPO_CODE_DIR}
    # LATERALITY_DISC_LEFT_IS: which eye a left-of-centre optic disc means in Tianjin's images. None =
    # the script's default ('OD', what every earlier run used), unless notebook 11's EyePACS validation
    # is clearly reliable (>= 80 % accuracy). Check a few Tianjin images of known right eyes by eye and
    # set 'OD' or 'OS' here -- that is the real answer (after changing it, delete laterality_resolved_v2.csv
    # on Drive so it is recomputed).
    LATERALITY_DISC_LEFT_IS = None
    _lat_args = ''
    _b = None
    if os.path.isfile(LATERALITY_VALIDATION):
        import json as _json
        _b = _json.load(open(LATERALITY_VALIDATION))['best']
    if LATERALITY_DISC_LEFT_IS:
        _lat_args = f' --disc-left-is {LATERALITY_DISC_LEFT_IS}'
        print(f'Laterality: --disc-left-is {LATERALITY_DISC_LEFT_IS} (set by hand)')
    elif _b and _b['accuracy_on_decided'] >= 0.8:
        _lat_args = (f" --disc-left-is {_b['disc_left_is']} --margin {_b['margin']}"
                     + (' --legacy-estimator' if _b['estimator'] == 'legacy' else ''))
        print(f"Laterality settings from the EyePACS validation: {_lat_args} "
              f"(accuracy {_b['accuracy_on_decided']:.1%}, {_b['uncertain_rate']:.1%} uncertain)")
    else:
        print('\u26a0 Laterality: script defaults (--disc-left-is OD --margin 0.1) -- '
              + (f"the EyePACS validation is too weak to use ({_b['accuracy_on_decided']:.1%} accuracy). "
                 if _b else 'no EyePACS validation. ')
              + 'Set LATERALITY_DISC_LEFT_IS after checking a few Tianjin images by eye.')
    !python module1/resolve_eye_laterality.py --dataset-dir "{TIANJIN_DIR}"{_lat_args}
    os.makedirs(os.path.dirname(LATERALITY_DRIVE_PATH), exist_ok=True)
    shutil.copy(LATERALITY_LOCAL_PATH, LATERALITY_DRIVE_PATH)
    print(f'✓ Computed and persisted laterality_resolved.csv to Drive')


In [ ]:
# Locate Module 1 checkpoints (classifier + EX/MA + HE/SE if trained) -- shared by all 3
# apply_to_progression_data.py calls below.
import glob

SEG_DIR   = os.path.join(MODULE1_RUNS, 'segmentation')
# Module 1 grader: 'eyepacs' (default) = DRG-Net's classifier trained on EyePACS (notebook 11, training
# capped at 3000 images/grade); 'fgadr' = the FGADR grader. Notebook 04_05's grader-comparison cell scores
# both on Tianjin's clinical grades -- set the one it recommends in notebooks 04_05, 06, 08, 09 and 10.
# With 'eyepacs' and no EyePACS grader on Drive, the FGADR one is used.
CLASSIFIER_RUN = 'eyepacs'
if not glob.glob(os.path.join(MODULE1_RUNS, 'classification', CLASSIFIER_RUN, 'saves', '*.pt')):
    print(f'\u26a0 No {CLASSIFIER_RUN} grader in {MODULE1_RUNS} yet -- using the FGADR one')
    CLASSIFIER_RUN = 'fgadr'
CLS_SAVES = os.path.join(MODULE1_RUNS, 'classification', CLASSIFIER_RUN, 'saves')
# Classifier input normalisation must match its training config (eyepacs_poc.yaml differs).
CLS_NORM_ARGS = ' --classifier-norm eyepacs' if CLASSIFIER_RUN == 'eyepacs' else ''

# EyePACS: best_validation_weights.pt (picked on the Kaggle public test, kept apart from its test set);
# FGADR: final_weights.pt (its "validation" set IS its test set, so the best-validation pick is optimistic).
_pref  = (['best_validation_weights.pt', 'final_weights.pt'] if CLASSIFIER_RUN == 'eyepacs'
          else ['final_weights.pt', 'best_validation_weights.pt'])
_cands = [os.path.join(CLS_SAVES, n) for n in _pref] + sorted(
    glob.glob(os.path.join(CLS_SAVES, '*.pt')), key=os.path.getmtime, reverse=True)
CLS_CKPT = next((p for p in _cands if os.path.isfile(p)), None)
assert CLS_CKPT, f"No classifier .pt in {CLS_SAVES} -- run notebook 02/03 (MODULE1_SOURCE='poc') or 11 ('drgnet' / 'drgnet_combined') first"

EX_CKPT = os.path.join(SEG_DIR, SEG_MODEL_PREFIX + 'ex', 'model_2.pth.tar')
MA_CKPT = os.path.join(SEG_DIR, SEG_MODEL_PREFIX + 'ma', 'model_2.pth.tar')
for _p in (CLS_CKPT, EX_CKPT, MA_CKPT):
    assert os.path.isfile(_p), f"missing checkpoint: {_p} -- run notebook 02/03 (MODULE1_SOURCE='poc') or 11 ('drgnet' / 'drgnet_combined') first"

HE_CKPT = os.path.join(SEG_DIR, SEG_MODEL_PREFIX + 'he', 'model_2.pth.tar')
SE_CKPT = os.path.join(SEG_DIR, SEG_MODEL_PREFIX + 'se', 'model_2.pth.tar')
print('classifier :', CLS_CKPT)
print('EX seg     :', EX_CKPT)
print('MA seg     :', MA_CKPT)
print('HE seg     :', HE_CKPT if os.path.isfile(HE_CKPT) else 'not trained yet')
print('SE seg     :', SE_CKPT if os.path.isfile(SE_CKPT) else 'not trained yet')

_extra_seg_args = ''
if os.path.isfile(HE_CKPT):
    _extra_seg_args += f' --seg-checkpoint HE="{HE_CKPT}"'
if os.path.isfile(SE_CKPT):
    _extra_seg_args += f' --seg-checkpoint SE="{SE_CKPT}"'

os.makedirs(MODULE1_CACHE_DIR, exist_ok=True)


In [ ]:
# Which grader for CLASSIFIER_RUN? Scores both graders (each with both of its saved weights) against
# Tianjin's CLINICAL grades, baseline + 2-year follow-up -- the images the pipeline grades. Their own
# test-set numbers in notebook 11 can't be compared (different datasets; FGADR's test set also picks its
# epoch). Primary view = per patient: worse-eye clinical grade vs. the highest grade given to that
# patient's images (needs no left/right); the per-eye view depends on the laterality file. ~10 min on a
# T4. Saved to Drive; delete the JSON to re-run. Then set the recommended grader in notebooks 04_05
# (cell above -- before the cache cells below), 06, 08, 09 and 10.
RUN_GRADER_COMPARISON = True
GRADER_COMPARISON_JSON = os.path.join(MODULE1_CACHE_DIR, 'grader_comparison_tianjin.json')
_cls_root = os.path.join(MODULE1_RUNS, 'classification')
_graders = {'fgadr': 'fgadr/saves/final_weights.pt', 'fgadr_bestval': 'fgadr/saves/best_validation_weights.pt',
            'eyepacs': 'eyepacs/saves/best_validation_weights.pt', 'eyepacs_final': 'eyepacs/saves/final_weights.pt'}
_grader_args = ''.join(f' --grader {n}="{os.path.join(_cls_root, f)}"' for n, f in _graders.items()
                       if os.path.isfile(os.path.join(_cls_root, f)))
if RUN_GRADER_COMPARISON and not os.path.isfile(GRADER_COMPARISON_JSON):
    %cd {REPO_CODE_DIR}
    !python module1/compare_graders_tianjin.py --tianjin-dir "{TIANJIN_DIR}"{_grader_args} --out "{GRADER_COMPARISON_JSON}"
if os.path.isfile(GRADER_COMPARISON_JSON):
    import json as _json
    _gc = _json.load(open(GRADER_COMPARISON_JSON))
    for _n, _r in _gc['graders'].items():
        _m = _r['patient']['both']
        print(f"{_n:<15} patient-level, both visits: kappa {(_m.get('quadratic_kappa') or 0):.3f}  "
              f"accuracy {_m['accuracy']:.3f}  within-one {_m['within_one_grade']:.3f}  (n={_m['n']})")
    _rec = (_gc['summary']['recommended'] or '').split('_')[0]
    print(f"\nRecommended: CLASSIFIER_RUN = '{_rec}'  (currently '{CLASSIFIER_RUN}')")
    if _rec and _rec != CLASSIFIER_RUN:
        print('\u26a0 Set it in the checkpoint cell above and re-run that cell before building the caches below '
              '(and in notebooks 06, 08, 09, 10).')

In [ ]:
# This branch's module1/apply_to_progression_data.py already has --checkpoint-every/--resume
# (so a long run survives a lost Colab session), plus the per-lesion biomarker stats and options
# notebooks 09-11 need. The manual upload of a local copy is therefore OFF by default: an older
# local copy would replace this branch's version for this session and drop those features.
# Only set this to True if your local copy has fixes that were never pushed.
UPLOAD_LOCAL_COPY = False

if UPLOAD_LOCAL_COPY:
    from google.colab import files
    uploaded = files.upload()  # pick module1/apply_to_progression_data.py
    for fname in uploaded:
        if fname != 'apply_to_progression_data.py':
            print(f'Skipping unexpected file: {fname}')
            continue
        dest = os.path.join(REPO_CODE_DIR, 'module1', fname)
        shutil.move(fname, dest)
        print(f'Overwrote {dest}')

!python {REPO_CODE_DIR}/module1/apply_to_progression_data.py --help | grep -E "checkpoint-every|resume"


In [ ]:
# FIRE
%cd {REPO_CODE_DIR}/module1
!python apply_to_progression_data.py \
    --images-dir /content/data/FIRE_dataset/FIRE/Images \
    --classifier-checkpoint "{CLS_CKPT}"{CLS_NORM_ARGS} \
    --seg-checkpoint EX="{EX_CKPT}" --seg-checkpoint MA="{MA_CKPT}"{_extra_seg_args} \
    --checkpoint-every 50 --resume \
    --out "{MODULE1_CACHE_DIR}/module1_outputs_fire.pt"


In [ ]:
# LongDR
!python apply_to_progression_data.py \
    --images-dir /content/data/LongDRScreening_20150209/FundusImagesNormalized \
    --classifier-checkpoint "{CLS_CKPT}"{CLS_NORM_ARGS} \
    --seg-checkpoint EX="{EX_CKPT}" --seg-checkpoint MA="{MA_CKPT}"{_extra_seg_args} \
    --checkpoint-every 50 --resume \
    --out "{MODULE1_CACHE_DIR}/module1_outputs_longdr.pt"


In [ ]:
# Tianjin -- the slow one. If this gets interrupted, just re-run this exact cell: --resume
# means it picks up where it left off instead of starting over.
!python apply_to_progression_data.py \
    --images-dir "{TIANJIN_DIR}/baseline fundus images" \
    --classifier-checkpoint "{CLS_CKPT}"{CLS_NORM_ARGS} \
    --seg-checkpoint EX="{EX_CKPT}" --seg-checkpoint MA="{MA_CKPT}"{_extra_seg_args} \
    --checkpoint-every 50 --resume \
    --out "{MODULE1_CACHE_DIR}/module1_outputs_tianjin.pt"


## Done

All 3 caches are now on Drive under `module1_cache_drgnet_comb/` (`MODULE1_SOURCE='drgnet_combined'`),
`module1_cache_drgnet/` (`'drgnet'`) or `module1_cache/` (`'poc'`). Next: notebook 09 (EyePACS pretraining), then 06 with the same
`MODULE1_SOURCE`, then 07, 08, 10.

## Before closing the runtime

Files written to Drive upload in the background. Run this last cell and wait for the ✓ before
disconnecting or deleting the runtime -- otherwise large files written at the end (model weights,
caches) can be lost.

In [ ]:
from google.colab import drive
drive.flush_and_unmount()
print('✓ Everything is uploaded to Google Drive -- safe to disconnect the runtime.')